In [1]:
from model.ctabgan import CTABGAN
from model.eval.evaluation import get_utility_metrics, stat_sim, privacy_metrics
import numpy as np
import pandas as pd
import glob

In [2]:
num_exp = 5
dataset = "MimicIV"
real_path = "Real_Datasets/MimicIV.csv"
fake_file_root = "Fake_Datasets"

In [ ]:
synthesizer = CTABGAN(
    raw_csv_path=real_path,
    test_ratio=0.20,

    categorical_columns=[
        "gender",
        "hospital_expire_flag"
    ],

    log_columns=[],     # none unless you want to log-transform skewed labs

    mixed_columns={},   # none unless you confirm heavy zero-mass columns

    general_columns=[
        "heart_rate_min", "heart_rate_max", "heart_rate_mean",
        "sbp_min", "sbp_max", "sbp_mean",
        "dbp_min", "dbp_max", "dbp_mean",
        "resp_rate_min", "resp_rate_max", "resp_rate_mean",
        "temperature_min", "temperature_max", "temperature_mean",
        "spo2_min", "spo2_max", "spo2_mean",
        "creatinine_min", "creatinine_max",
        "sodium_min", "sodium_max",
        "potassium_min", "potassium_max",
        "hemoglobin_min", "hemoglobin_max",
        "wbc_min", "wbc_max"
    ],

    non_categorical_columns=[],  # since we dropped IDs

    integer_columns=[
        "age_at_intime"
    ],

    problem_type={"Classification": "hospital_expire_flag"}
)

for i in range(num_exp):
    synthesizer.fit()
    syn = synthesizer.generate_samples()
    syn.to_csv(fake_file_root+"/"+dataset+"/"+ dataset+"_fake_{exp}.csv".format(exp=i), index= False)

 34%|███▍      | 51/150 [4:29:19<4:58:34, 180.95s/it] 

In [ ]:
fake_paths = glob.glob(fake_file_root+"/"+dataset+"/"+"*")

In [ ]:
model_dict =  {"Classification":["lr","dt","rf","mlp","svm"]}
result_mat = get_utility_metrics(real_path,fake_paths,"MinMax",model_dict, test_ratio = 0.20)

result_df  = pd.DataFrame(result_mat,columns=["Acc","AUC","F1_Score"])
result_df.index = list(model_dict.values())[0]
result_df

In [ ]:
thyroid_categorical = ['Class']
stat_res_avg = []
for fake_path in fake_paths:
    stat_res = stat_sim(real_path,fake_path,thyroid_categorical)
    stat_res_avg.append(stat_res)

stat_columns = ["Average WD (Continuous Columns","Average JSD (Categorical Columns)","Correlation Distance"]
stat_results = pd.DataFrame(np.array(stat_res_avg).mean(axis=0).reshape(1,3),columns=stat_columns)
stat_results

In [ ]:
priv_res_avg = []
for fake_path in fake_paths:
    priv_res = privacy_metrics(real_path,fake_path)
    priv_res_avg.append(priv_res)
    
privacy_columns = ["DCR between Real and Fake (5th perc)","DCR within Real(5th perc)","DCR within Fake (5th perc)","NNDR between Real and Fake (5th perc)","NNDR within Real (5th perc)","NNDR within Fake (5th perc)"]
privacy_results = pd.DataFrame(np.array(priv_res_avg).mean(axis=0).reshape(1,6),columns=privacy_columns)
privacy_results

In [ ]:
import os
import pandas as pd

from Evaluation.cdf_tail_metrics import CDFTailMetrics
from Evaluation.support_coverage import SupportCoverage
from Evaluation.rare_event_recall import RareEventRecall

n_experiments = 5

cdf_eval = CDFTailMetrics(label_col="Class", tau=0.9)
sc_eval  = SupportCoverage(label_col="Class", n_bins=5, rare_threshold=0.01, include_label_in_combo=True)
rer_eval = RareEventRecall(label_col="Class")

rows = []

for exp in range(n_experiments):
    syn_path = os.path.join(fake_file_root, dataset, f"{dataset}_fake_{exp}.csv")
    if not os.path.exists(syn_path):
        print("Missing:", syn_path)
        continue

    res = {"exp": exp, "syn_path": syn_path}
    res.update(cdf_eval.evaluate_paths(real_path, syn_path))
    res.update(sc_eval.evaluate_paths(real_path, syn_path))
    res.update(rer_eval.evaluate_paths(real_path, syn_path))
    rows.append(res)

results_df = pd.DataFrame(rows)
results_df
